# House Price Prediction Model
**ITI AI Track - Level 1 Capstone Project**

This notebook contains the complete machine learning workflow:
1. Data loading and initial inspection
2. Exploratory Data Analysis (EDA)
3. Data cleaning and feature engineering
4. Preprocessing pipeline (`ColumnTransformer`)
5. Model training & evaluation (Linear Regression vs Random Forest)
6. Exporting the trained model and metadata

## 1. Import Libraries

In [ ]:
import os
import re
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn tools
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set plot style
sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")

## 2. Load and Inspect Dataset

In [ ]:
# Load dataset
data_path = "../data/house_prices.csv" if os.path.exists("../data/house_prices.csv") else "house_prices.csv"
df = pd.read_csv(data_path)

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
# Check column data types and non-null counts
df.info()

In [ ]:
# Check missing values
df.isnull().sum()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Helper parsing functions for preliminary EDA visualization
def parse_price(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    if 'Cr' in val:
        m = re.search(r'([\d\.]+)', val)
        return float(m.group(1)) * 1e7 if m else np.nan
    elif 'Lac' in val or 'Lakh' in val:
        m = re.search(r'([\d\.]+)', val)
        return float(m.group(1)) * 1e5 if m else np.nan
    return np.nan

def parse_area(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r'([\d\.]+)', str(val))
    if not m:
        return np.nan
    num = float(m.group(1))
    if 'sqm' in str(val).lower():
        return num * 10.764
    return num

# Create sample dataframe for EDA plots
df_eda = df.copy()
df_eda['price'] = df_eda['Amount(in rupees)'].apply(parse_price)
df_eda['area_sqft'] = df_eda['Carpet Area'].apply(parse_area)
df_eda = df_eda.dropna(subset=['price', 'area_sqft'])
df_eda = df_eda[(df_eda['price'] > 1e5) & (df_eda['area_sqft'] >= 200) & (df_eda['area_sqft'] <= 6000)]

### Plot 1: Target Price Distribution (Original vs Log Transform)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Original Price Distribution
sns.histplot(df_eda['price'] / 1e5, kde=True, ax=ax[0], color='steelblue', bins=35)
ax[0].set_title('Price Distribution (in Lacs)', fontsize=12)
ax[0].set_xlabel('Price (Lacs)')

# Log Transformed Price Distribution
sns.histplot(np.log1p(df_eda['price']), kde=True, ax=ax[1], color='forestgreen', bins=35)
ax[1].set_title('Log-Transformed Price log1p(Price)', fontsize=12)
ax[1].set_xlabel('log(Price)')

plt.tight_layout()
plt.show()

### Plot 2: Carpet Area vs Price

In [ ]:
plt.figure(figsize=(9, 5))
sample = df_eda.sample(n=min(2500, len(df_eda)), random_state=42)
sns.scatterplot(data=sample, x='area_sqft', y=sample['price']/1e5, alpha=0.4, color='darkorange')
plt.title('Property Area (sqft) vs Price (Lacs)', fontsize=12)
plt.xlabel('Carpet Area (sqft)')
plt.ylabel('Price (Lacs)')
plt.show()

### Plot 3: Top 15 Locations by Average Price

In [ ]:
top_locations = df_eda.groupby('location')['price'].mean().sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 5))
sns.barplot(x=top_locations.values / 1e5, y=top_locations.index, palette='mako')
plt.title('Top 15 Locations by Average Price (Lacs)', fontsize=12)
plt.xlabel('Average Price (Lacs)')
plt.ylabel('Location')
plt.show()

### Plot 4: Price vs Furnishing & Bathrooms

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Furnishing vs Price
sns.boxplot(data=df_eda[df_eda['price'] < 3e7], x='Furnishing', y=df_eda['price']/1e5, ax=ax[0], palette='Set2')
ax[0].set_title('Price by Furnishing Status', fontsize=12)
ax[0].set_ylabel('Price (Lacs)')

# Bathrooms vs Price
sns.barplot(data=df_eda[df_eda['Bathroom'].between(1, 5)], x='Bathroom', y=df_eda['price']/1e5, ax=ax[1], palette='Blues_d')
ax[1].set_title('Average Price by Number of Bathrooms', fontsize=12)
ax[1].set_ylabel('Average Price (Lacs)')

plt.tight_layout()
plt.show()

## 4. Data Cleaning and Feature Engineering

In [ ]:
# Cleaning functions
def clean_amount_full(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    if "Call for Price" in val or "Price on Request" in val:
        return np.nan
    cr_match = re.search(r"([\d\.]+)\s*Cr", val, re.IGNORECASE)
    if cr_match:
        return float(cr_match.group(1)) * 1e7
    lac_match = re.search(r"([\d\.]+)\s*(?:Lac|Lakh|L)", val, re.IGNORECASE)
    if lac_match:
        return float(lac_match.group(1)) * 1e5
    clean_num = re.sub(r"[^\d\.]", "", val)
    return float(clean_num) if clean_num else np.nan

def clean_floor_num(val):
    if pd.isna(val):
        return 1
    val = str(val).strip().lower()
    if "ground" in val or "lower" in val:
        return 0
    if "basement" in val:
        return -1
    m = re.search(r"^(\d+)", val)
    return int(m.group(1)) if m else 1

def extract_int(val, default=1):
    if pd.isna(val):
        return default
    m = re.search(r"(\d+)", str(val))
    return int(m.group(1)) if m else default

In [ ]:
# Apply data cleaning
df_clean = df.copy()

# 1. Clean target price
df_clean['price_rupees'] = df_clean['Amount(in rupees)'].apply(clean_amount_full)
df_clean = df_clean.dropna(subset=['price_rupees'])
df_clean = df_clean[df_clean['price_rupees'] >= 100000]

# 2. Clean carpet area & fallback to super area
df_clean['carpet_area_sqft'] = df_clean['Carpet Area'].apply(parse_area)
df_clean['super_area_sqft'] = df_clean['Super Area'].apply(parse_area)
df_clean['carpet_area_sqft'] = df_clean['carpet_area_sqft'].fillna(df_clean['super_area_sqft'])
df_clean = df_clean.dropna(subset=['carpet_area_sqft'])
df_clean = df_clean[(df_clean['carpet_area_sqft'] >= 200) & (df_clean['carpet_area_sqft'] <= 10000)]

# 3. Clean floor number, bathrooms, balconies
df_clean['floor_num'] = df_clean['Floor'].apply(clean_floor_num)
df_clean['bathrooms'] = df_clean['Bathroom'].apply(lambda x: extract_int(x, default=2))
df_clean['balconies'] = df_clean['Balcony'].apply(lambda x: extract_int(x, default=1))

# 4. Group high-cardinality locations (Top 50 + other)
df_clean['location_clean'] = df_clean['location'].astype(str).str.strip().str.lower()
top_50_locs = df_clean['location_clean'].value_counts().head(50).index.tolist()
df_clean['location_clean'] = df_clean['location_clean'].apply(lambda x: x if x in top_50_locs else 'other')

# 5. Fill categorical missing values
df_clean['furnishing'] = df_clean['Furnishing'].fillna('Unfurnished').astype(str).str.strip()
df_clean['transaction'] = df_clean['Transaction'].fillna('Resale').astype(str).str.strip()

# 6. Remove extreme price-per-sqft outliers (1st to 99th percentile)
df_clean['price_per_sqft'] = df_clean['price_rupees'] / df_clean['carpet_area_sqft']
q_low = df_clean['price_per_sqft'].quantile(0.01)
q_high = df_clean['price_per_sqft'].quantile(0.99)
df_clean = df_clean[(df_clean['price_per_sqft'] >= q_low) & (df_clean['price_per_sqft'] <= q_high)]

print(f"Cleaned dataset rows: {df_clean.shape[0]}")
df_clean[['location_clean', 'carpet_area_sqft', 'floor_num', 'furnishing', 'bathrooms', 'price_rupees']].head()

## 5. Pipeline Setup & Train-Test Split

In [ ]:
# Define features and target
feature_cols = ['location_clean', 'carpet_area_sqft', 'floor_num', 'furnishing', 'transaction', 'bathrooms', 'balconies']
X = df_clean[feature_cols].rename(columns={'location_clean': 'location'})
y = df_clean['price_rupees']

# Train/Test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessor ColumnTransformer
numeric_features = ['carpet_area_sqft', 'floor_num', 'bathrooms', 'balconies']
categorical_features = ['location', 'furnishing', 'transaction']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_features)
    ]
)

# Apply log1p transform to target variable
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)
print("Preprocessing pipeline initialized successfully.")

## 6. Model Training and Comparison

In [ ]:
# 1. Linear Regression (Baseline)
lr_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])
lr_pipe.fit(X_train, y_train_log)
lr_preds = np.expm1(lr_pipe.predict(X_test))

lr_mae = mean_absolute_error(y_test, lr_preds)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_r2 = r2_score(y_test, lr_preds)

# 2. Random Forest Regressor (Advanced)
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1))
])
rf_pipe.fit(X_train, y_train_log)
rf_preds = np.expm1(rf_pipe.predict(X_test))

rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_r2 = r2_score(y_test, rf_preds)

# Summary comparison table
summary_df = pd.DataFrame({
    'Model': ['Linear Regression (Baseline)', 'Random Forest Regressor'],
    'MAE (Rupees)': [f"₹ {lr_mae:,.2f}", f"₹ {rf_mae:,.2f}"],
    'RMSE (Rupees)': [f"₹ {lr_rmse:,.2f}", f"₹ {rf_rmse:,.2f}"],
    'R2 Score': [f"{lr_r2:.4f}", f"{rf_r2:.4f}"]
})
summary_df

### Predicted vs Actual Scatter Plot

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test/1e5, y=rf_preds/1e5, alpha=0.4, color='dodgerblue')
max_val = max(y_test.max()/1e5, rf_preds.max()/1e5)
plt.plot([0, max_val], [0, max_val], '--r', linewidth=2, label='Perfect Prediction (y=x)')
plt.title('Actual vs Predicted House Prices (Random Forest)', fontsize=12)
plt.xlabel('Actual Price (Lacs)')
plt.ylabel('Predicted Price (Lacs)')
plt.legend()
plt.show()

## 7. Export Best Model and Metadata

In [ ]:
# Export best pipeline
best_pipeline = rf_pipe if rf_r2 > lr_r2 else lr_pipe
os.makedirs('../models', exist_ok=True)

# 1. Save trained pipeline
joblib.dump(best_pipeline, '../models/house_price.pkl')
print("Model pipeline saved to ../models/house_price.pkl")

# 2. Save supported locations list
export_locations = sorted(list(set(top_50_locs + ['other'])))
with open('../models/locations.json', 'w', encoding='utf-8') as f:
    json.dump(export_locations, f, indent=2, ensure_ascii=False)
print(f"Exported {len(export_locations)} locations to ../models/locations.json")

### Verification / Single Sample Inference

In [ ]:
# Test sample prediction
sample_input = pd.DataFrame([{
    'location': 'thane',
    'carpet_area_sqft': 1200,
    'floor_num': 4,
    'furnishing': 'Semi-Furnished',
    'transaction': 'Resale',
    'bathrooms': 2,
    'balconies': 1
}])

loaded_pipe = joblib.load('../models/house_price.pkl')
pred_price = np.expm1(loaded_pipe.predict(sample_input)[0])
print(f"Sample Predicted Price: ₹ {pred_price:,.2f} ({pred_price/1e5:.2f} Lacs)")